In [1]:
import os
from pydantic import BaseModel , Field
from typing import TypedDict, Annotated, Sequence , Literal
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

C:\Users\Syed Ameer Baji\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class AGENTSTATE(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    route : str 


In [3]:
model = ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY)

In [4]:
class RouterDecision(BaseModel):
    """
        This is an simple Pydantic scheme for AI agnet router decision 
        with few defined decisions
    """
    route : Literal["math","general"] = Field(description = "choose math for calculation , general otherwise")    
    
router_model = model.with_structured_output(RouterDecision)

def router(state : AGENTSTATE):
    sys_prompt = SystemMessage(
        content=(
            "Classify the user's latest request. "
            "Choose 'math' for arithmetic or calculations. "
            "Choose 'general' for all other requests."
        )
    )
    decision = router_model.invoke([sys_prompt , *state['messages']])
    # in the defined "RouterDecision" we have route 
    return {
        "route":decision.route
    }    
    

In [5]:
@tool
def add(x: int, y: int) -> int:
    """
        Add two numbers x and y returns x + y
    """
    return x + y


@tool
def sub(x: int, y: int) -> int:
    """
        subtract two numbers x and y returns x - y
    """
    return x - y


@tool
def multiple(x: int, y: int) -> int:
    """
        multiple two numbers x and y returns x * y
    """
    return x * y

math_model = model.bind_tools(tools = [add , sub , multiple])

In [6]:
def general_agent(state: AGENTSTATE):
    """
        This Agent solves the General questions 
    """
    system_prompt = SystemMessage(
        content="You are an excellend AI math assistant you are required to answer the following questions simply ")
    res = model.invoke([system_prompt] + state['messages'])
    
    # this will be passed to the next nodes
    # as the agent state memory holds a sequence of list of messages
    # so return the list of result
    return {
        "messages": [res]
    }




def math_agent(state : AGENTSTATE):
    """
    This Agent uses tools to solve the question 
    """    
    sys_pro = SystemMessage(
            content=(
                "You are a math assistant. Use the provided tools "
                "for calculations. For sequential operations, "
                "use each result in the next operation."
            )
        )
    res = math_model.invoke([sys_pro, *state['messages']])
    return {
        "messages":[res]
    }                
    
    
def should_continue_or_not(state : AGENTSTATE) -> str:    
    """
    Each after every step we check wheather to call the tools
    or not or to END it    
    """
    _tool_calls_ = getattr(state['messages'][-1], "tool_calls", [])
    if not _tool_calls_ :
        return "end"
    return "continue"

In [7]:
graph = StateGraph(state_schema = AGENTSTATE)
graph.add_node("router", router)
graph.add_node("math", math_agent)
graph.add_node("general", general_agent)

graph.add_node("tools", ToolNode(tools=[add , sub , multiple]))

graph.set_entry_point("router")
graph.add_conditional_edges(
    "router",
    lambda state : state['route'],
    {
        "math":"math",
        "general":"general"
        
    }
)
graph.add_conditional_edges("math",
    should_continue_or_not,{
        "continue":"tools",
        "end":END
    }
)
graph.add_edge("tools", "math")
graph.add_edge("general", END)

app = graph.compile()

In [8]:
res = app.invoke({
    "messages":[
        HumanMessage(content="add 1,1 and then subtract 1 then mulitply with 10")
        ]
})
                

In [14]:
YELLOW = "\033[33m"

for msg in res['messages']:
    print(f">> {msg.__class__.__name__} : {msg.content}")
    for tool_call in getattr(msg, "tool_calls", []):
        print(f"\t\t{YELLOW}{tool_call['name']}({tool_call['args']})\033[0m")

>> HumanMessage : add 1,1 and then subtract 1 then mulitply with 10
>> AIMessage : 
		add({'x': 1, 'y': 1})
>> ToolMessage : 2
>> AIMessage : 
		sub({'x': 2, 'y': 1})
>> ToolMessage : 1
>> AIMessage : 
		multiple({'x': 1, 'y': 10})
>> ToolMessage : 10
>> AIMessage : The final result is **10**.
